In [67]:
import os
from openai import OpenAI
import pandas as pd
import numpy as np
import re
from pypinyin import lazy_pinyin
from rapidfuzz import fuzz
import math
from uuid import uuid4 as uuid
from dotenv import load_dotenv
import subprocess
from tqdm import tqdm
import json
load_dotenv(".env")

root = "/mnt/NextcloudSacmData/sacm.av/files/Recordings"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
df = pd.read_csv("songs.csv")
df.tail(1)

,code,type,title,lyrics,pinyin
496,UNK-10,Hymn,主活在我心,从前在罪中远离神，\n心怀毫无亮光。\n从主言语中才觉悟，\n基督活在心中。\n\n主活在我...,cong qian zai zui zhong yuan li shen xin huai ...


In [ ]:
files = sorted([f"{d}/{f}" for d in os.listdir(root) for f in os.listdir(f"{root}/{d}")])

to_rename = {}
for i, row in df.iterrows():
    title = row.title
    title_ = re.sub(r"[，。！？、“”：；\n]", "", row.title).replace("赞美诗24", "")
    if title_ == row.title:
        continue
    df.at[i, "title"] = title_
    if not (affected := [f for f in files if title in f]):
        continue
    print(f">>>>>>>>>> {row.name}: {row.title}")
    print("\n".join(affected))
    for f in affected:
        new_name = to_rename.get(f, f).replace(title, title_)
        to_rename[f] = new_name

In [ ]:
folders = set()
for old_path, new_path in to_rename.items():
    d, _ = old_path.split("/", 1)
    folders.add(d)
    os.rename(f"{root}/{old_path}", f"{root}/{new_path}")

# for d in folders:
#     !sudo -u www-data php /var/www/html/nextcloud_sacm/occ files:scan --path sacm.av/files/Recordings/{d}

In [ ]:
def pinyin(text):
    text = re.sub(r"[，。！？、“”：；\n]", " ", text)
    result = " ".join(lazy_pinyin(text))
    return re.sub(r"\s+", " ", result).strip()


def windows(tokens, size, step):
    if len(tokens) <= size:
        yield " ".join(tokens)
    else:
        for i in range(0, len(tokens) - size + 1, step):
            yield " ".join(tokens[i:i+size])


def best_window_score(query_py, lyrics_py, size=50, step=10):
    query_tokens = query_py.split()
    lyric_tokens = lyrics_py.split()
    score = max(
        fuzz.ratio(qw, lw)
        for qw in windows(query_tokens, size, step)
        for lw in windows(lyric_tokens, size, step)
    )
    return score / 100


def get_duration(filepath) -> float:
    result = subprocess.run(
        [
            "ffprobe",
            "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            filepath,
        ],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(f"[ERROR] ffprobe failed: {result.stderr}")
        return 0
    return float(result.stdout.strip())


def split_to_limit(filepath, limit=26_214_400, margin=0.90, out_dir="tmp"):
    size = os.path.getsize(filepath)
    if size <= limit:
        return [filepath]
    os.makedirs(out_dir, exist_ok=True)
    duration = get_duration(filepath)
    bitrate_kbps = 128
    chunk_seconds = max(1, int(limit * margin * 8 / (bitrate_kbps * 1000)))
    chunk_paths = []
    
    for start in range(0, math.ceil(duration), chunk_seconds):
        chunk_path = os.path.join(out_dir, f"{uuid()}.mp3")
        subprocess.run([
            "ffmpeg",
            "-y",
            "-ss", str(start),
            "-t", str(chunk_seconds),
            "-i", filepath,
            "-vn",
            "-c:a", "libmp3lame",
            "-b:a", f"{bitrate_kbps}k",
            chunk_path,
        ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        chunk_paths.append(chunk_path)
        print(
            f"chunk {len(chunk_paths)}: "
            f"{os.path.getsize(chunk_path):,} bytes "
            f"(limit {limit:,}) -> {chunk_path}"
        )
    return chunk_paths

In [ ]:
def match_zoom_to_sq(d, tol=10, verbose=False):
    files = sorted(os.listdir(f"{root}/{d}"))
    zoom_files = [f for f in files if f.startswith("ZOOM")]
    sq_files = [f for f in files if not f.startswith("ZOOM")]
    
    if not zoom_files or not sq_files:
        return {}

    durations = {f: int(get_duration(f"{root}/{d}/{f}")) for f in files}
    if verbose:
        print(json.dumps(durations, indent=2, ensure_ascii=False))

    reduced_sq_files = []
    for f in sq_files:
        other_files = [x for x in reduced_sq_files if not x.startswith("SQ")]
        if f.startswith("SQ") or not any(durations[f] == durations[x] for x in other_files):
            reduced_sq_files.append(f)
    sq_files = reduced_sq_files

    if verbose:
        print(zoom_files)
        print(sq_files)

    # If same number of files, just pair in order
    if len(zoom_files) == len(sq_files):
        ordered_matches = dict(zip(zoom_files, sq_files))
        # Make sure the sizes match though
        if all(abs(durations[z] - durations[s]) < tol for z, s in ordered_matches.items()):
            return ordered_matches

    # Otherwise, take the largest filesize (P&W) pair as reference
    matches = {}

    def traverse(ref_zoom_idx, ref_sq_idx, zoom_list, sq_list):
        z_idx = ref_zoom_idx + 1
        s_idx = ref_sq_idx + 1
        while z_idx < len(zoom_list) and s_idx < len(sq_list):
            zoom_file = zoom_list[z_idx]
            for i, sq_file in enumerate(sq_list):
                if sq_file.startswith("SQ") and i < s_idx:
                    continue
                if verbose:
                    print(f"{i=} {s_idx=} {zoom_file} ({durations[zoom_file]}) : {sq_file} ({durations[sq_file]})")
                if abs(durations[zoom_file] - durations[sq_file]) > tol:
                    continue
                matches[zoom_file] = sq_file
                if sq_file.startswith("SQ"):
                    s_idx = i + 1
                break
            z_idx += 1

    def prune_same_values(d):
        counts = {v: len([k for k in d if d[k] == v]) for v in d.values()}
        return {k: v for k, v in d.items() if counts[v] == 1}

    largest_zoom_file = max(zoom_files, key=lambda f: durations[f])
    largest_sq_file = max(sq_files, key=lambda f: durations[f])
    if not largest_sq_file.startswith("SQ"):
        # since order can't be gleaned from filename, just traverse the whole list
        traverse(-1, -1, zoom_files, sq_files)
        return prune_same_values(matches)

    if abs(durations[largest_zoom_file] - durations[largest_sq_file]) <= tol:
        matches[largest_zoom_file] = largest_sq_file
        zoom_idx = zoom_files.index(largest_zoom_file)
        sq_idx = sq_files.index(largest_sq_file)
    else:
        cost = np.array([
            [abs(durations[z] - durations[s]) for s in sq_files]
            for z in zoom_files
        ])
        zoom_idx, sq_idx = np.unravel_index(np.argmin(cost), cost.shape)
        matches[zoom_files[zoom_idx]] = sq_files[sq_idx]

    traverse(zoom_idx, sq_idx, zoom_files, sq_files)
    traverse(len(zoom_files) - zoom_idx - 1, len(sq_files) - sq_idx - 1, zoom_files[::-1], sq_files[::-1])

    return prune_same_values(matches)

# matches = match_zoom_to_sq("2026-04-11")
# print(json.dumps(matches, indent=2, ensure_ascii=False))

In [ ]:
lyrics = """哦~ 神伟大的爱
何其长阔高深
竟不吝惜祂独生爱子
为我们众人舍了
神称为义的人谁能控告
谁能定他们的罪
神若帮助我们 谁能敌挡我们
谁能使我与神的爱隔绝

难道是患难吗 是困苦吗
是逼迫吗 是饥饿吗
是赤身露体 危险 刀剑吗
不论是高处的 是低处的
现在的事 将来的事
都不能叫我与神的爱隔绝"""
num = max([int(code.split("-")[1]) for code in df.loc[df.code.str.startswith("UNK")].code])
df.loc[len(df)] = {
    "code": f"UNK-{num + 1}",
    "type": "Hymn",
    "title": "主活在我心",
    "lyrics": lyrics,
    "pinyin": pinyin(lyrics),
}
# idx = 495
# df.loc[idx, "lyrics"] = lyrics
# df.loc[idx, "pinyin"] = pinyin(lyrics)
df.to_csv("songs.csv", index=False)
df.tail(1)

,code,type,title,lyrics,pinyin
496,UNK-10,Hymn,主活在我心,从前在罪中远离神，\n心怀毫无亮光。\n从主言语中才觉悟，\n基督活在心中。\n\n主活在我...,cong qian zai zui zhong yuan li shen xin huai ...


In [3]:
for d in sorted(os.listdir(root), reverse=True):
    if not d.startswith("2025"):
        continue
    files = sorted(os.listdir(f"{root}/{d}"))
    for f in files:
        if bool(re.search(r'[\u4e00-\u9fff]', f)):
            continue
        filepath = f"{root}/{d}/{f}"
        duration = get_duration(filepath)
        if duration > 60:
            mins, secs = int(duration // 60), int(duration % 60)
            print(f"{filepath} - {mins:02d}:{secs:02d}")

In [ ]:
lyrics = ""
for filepath in tqdm(split_to_limit(f"{root}/2025-05-10/SQ-ST404_是你的爱.mp3")):
    print(filepath)
    audio_file = open(filepath, "rb")
    transcription = client.audio.transcriptions.create(
        # model="gpt-4o-transcribe", 
        model="whisper-1", 
        file=audio_file,
        language="zh",
    )
    lyrics += transcription.text
len(lyrics), lyrics

100%|██████████| 1/1 [00:27<00:00, 27.81s/it]


(1432,
 '由 Amara.org 社群提供的字幕 從天父而來的愛恨天 把我們冰冷的心溶解 讓我們獻出每個音符 把它化為讚美之泉 從天父而來的愛恨天 把我們冰冷的心溶解 讓我們獻出每個音符 把它化為讚美之泉 讓我們張開口 舉起手 向永生之主呈現 使讚美之泉流入 每個人的心間 讓我們張開口 舉起手 向永生之主呈現 使讚美之泉流入 每個人的心間 讓我們張開口 舉起手 向永生之主呈現 使讚美之泉流入 每個人的心間 我願意相扶 我願意相扶 在你愛的懷抱中 我願意相扶 你是我的主 你是我的主 永遠在你懷抱中 你是我 你是我的主 獅子架上的光芒 溫柔又思想 帶著主愛的力量 向著我照亮 我的心不再隱藏 完全地擺上 願主愛來教導我 在愛中的自由釋放 我願意相扶 我願意相扶 在你愛的懷抱中 我願意相扶 你是我的主 你是我的主 永遠在你懷抱中 你是我 你是我的主 我願意相扶 我願意相扶 我願意相扶 在你愛的懷抱中 我願意相扶 你是我的主 你是我的主 你是我的主 永遠在你懷抱中 你是我 你是我的主 永遠在你懷抱中 你是我 你是我的主 是 主耶穌是我們的主 在人生中 我們一定會遇到困難 有時候這些困難 我們會覺得好像是一個 很長期的挑戰 在這些時候 我們或許會覺得 我們正在低谷當中 而有時上帝好像很遙遠 很沉默 甚至懷疑上帝的愛 但經文告訴我們 上帝的愛是一份不離不棄的愛 不是因為我們配得 而只因他愛我們 在羅馬書8章32和35節說到 上帝既不顧惜自己的兒子 為我們眾人捨了他 豈不也把萬物和他 一同白白的賜給我們嗎 誰能使我們與基督的愛隔絕呢 難道是患難嗎 是困苦嗎 是迫害嗎 是飢餓嗎 是隻身漏體 是危險嗎 是刀劍嗎 接下來我們用以下的詩歌 繼續讚美主 我們若因聖靈感動 或有身體的需要 我們能站坐或跪著來敬拜 《聖靈感動》 詞曲 李宗盛 哦 神偉大的愛 何其誠惑高深 竟不領洗他獨身 愛自為我們眾人捨了 神成為一的人 誰能控告 誰能定他們的罪 誰能保住我們 誰能抵擋我們 誰能使我與神的愛隔絕 難道是患難嗎 是困苦嗎 是迫害嗎 是飢餓嗎 是隻身漏體 是危險嗎 是刀劍嗎 不論是高處的 是低處的 現在的事 將來的事 都不能將我與神的愛隔絕 哦 神偉大的愛 何其誠惑高深 竟不領洗他獨身 愛自為我們眾人捨了 神成為一的人 誰能控告 誰能定他們的罪 神若幫助我們 誰能抵擋我們 誰能使我與神的愛隔絕

In [ ]:
titles = {}
title_to_last_chunk_idx = {}

query_lyrics = re.sub(r"[，。！、\n]", " ", lyrics)

chunk_size = 120
for i, start in enumerate(range(0, len(query_lyrics), chunk_size)):
    chunk = query_lyrics[start:start + chunk_size]
    if len(chunk) < 50:
        continue
    query_py = pinyin(chunk)
    scores = [best_window_score(query_py, lyric_py, size=min(len(chunk), 100), step=5) for lyric_py in df.pinyin]
    best_idx = np.argmax(scores)
    best_title = df.iloc[best_idx]["title"]
    best_score = scores[best_idx]
    print(f"[{start}:{start+chunk_size}] {best_title=}, {best_score=}")
    if best_title in titles and (i - title_to_last_chunk_idx.get(best_title, -5)) <= 2:
        titles[best_title] = max(titles[best_title], best_score) * 1.2
    else:
        titles[best_title] = best_score
    title_to_last_chunk_idx[best_title] = i

print(f"{titles=}")
final_titles = [title for title, score in titles.items() if score > 0.7]
print(f"Songs: {'_'.join(final_titles)}")

[0:120] best_title='赞美之泉', best_score=0.6717325227963526
[120:240] best_title='爱，我愿意', best_score=0.6174999999999999
[240:360] best_title='爱，我愿意', best_score=0.8607594936708861
[360:480] best_title='爱，我愿意', best_score=0.5821474773609314
[480:600] best_title='伟大奇妙神', best_score=0.5586734693877551
[600:720] best_title='谁能使我与神的爱隔绝', best_score=0.6782376502002669
[720:840] best_title='谁能使我与神的爱隔绝', best_score=0.6177215189873417
[840:960] best_title='谁能使我与神的爱隔绝', best_score=0.7602649006622516
[960:1080] best_title='谁能使我与神的爱隔绝', best_score=0.8416988416988417
[1080:1200] best_title='谁能使我与神的爱隔绝', best_score=0.9022757697456493
[1200:1320] best_title='谁能使我与神的爱隔绝', best_score=0.6047745358090186
[1320:1440] best_title='以利亚的日子', best_score=0.5525291828793775
titles={'赞美之泉': 0.6717325227963526, '爱，我愿意': 1.2394936708860758, '伟大奇妙神': 0.5586734693877551, '谁能使我与神的爱隔绝': 1.6876723097463278, '以利亚的日子': 0.5525291828793775}
Songs: 爱，我愿意_谁能使我与神的爱隔绝


In [17]:
print(df.loc[df.title.str.contains("我的盼望在于祢")].iloc[0].lyrics)

耶稣因为你 我找到希望 你爱的真光 照亮我心房
有你在身旁 我不再迷茫 所有的惧怕 完全得释放

耶稣感谢你 带给我平安 挪去我忧伤 带我出黑暗
生命有了你 从此不一样 你给我翅膀 能自由飞翔

我的盼望 在于你耶稣 你爱照亮我前方路
带我出低谷 使我心满足 化一切咒诅变为祝福
一生仰望 美好的救主 靠你力量我得坚固
认识你耶稣 是最美的祝福
每一天紧紧跟随你的脚步


In [ ]:
chunk = query_lyrics[2640:2760]
print("Query:", chunk)
query_pinyin = pinyin(chunk)
for t in ["宁静谷"]:
    inds = df.loc[df.title == t].index
    for idx in inds:
        print(f"[{idx}] {t}: {df.pinyin[idx]}")
        fuzz_score = best_window_score(query_pinyin, df.pinyin[idx], size=100, step=3)
        print(f"{fuzz_score}")

Query:  我学会了信靠他 依靠他 有一次当我 向一位朋友 倾诉我的挣扎时 他推荐我 他推荐给我一首 藏民之群的歌 叫《宁静谷》 歌词中写道 生活中的仓促 生命里的难处 只愿向他来倾诉 平安祝福在这谷 我觉得这首歌 正好讲述了 那段时期 上帝如何 把
[77] 宁静谷: zai wo xin ling shen chu you yi zuo ning jing gu wo he wo qin ai de zhu zai qi zhong an ran man bu sheng huo zhong de cang cu sheng ming li de nan chu zhi yuan xiang ta lai qing su ping an zhu fu zai zhe gu wo yu wo zhu xiang yue zhi chu chang yang zhe fen ning jing an xiang jiu xiang shi zai tian tang wo yu wo zhu xiang yue zhi chu zhu ling wo guo si yin you gu shi wo xi le zou ren sheng lu
score=0.5467158003484595, fuzz_score=0.5852417302798982, 0.2868419756502333


In [ ]:
def get_titles(filepath):
    print(f"Processing {filepath}")

    cropped_paths = split_to_limit(filepath)
    lyrics = ""
    for filepath in tqdm(cropped_paths):
        audio_file = open(filepath, "rb")
        transcription = client.audio.transcriptions.create(
            # model="gpt-4o-transcribe", 
            model="whisper-1", 
            file=audio_file,
            language="zh",
        )
        lyrics += transcription.text

    if not lyrics:
        return []

    titles = {}
    title_to_last_chunk_idx = {}

    query_lyrics = re.sub(r"[，。！、\n]", " ", lyrics)

    chunk_size = 120
    for i, start in enumerate(range(0, len(query_lyrics), chunk_size)):
        chunk = query_lyrics[start:start + chunk_size]
        if len(chunk) < 50:
            continue
        query_py = pinyin(chunk)
        scores = [best_window_score(query_py, lyric_py, size=min(len(chunk), 100), step=5) for lyric_py in df.pinyin]
        best_idx = np.argmax(scores)
        best_title = df.iloc[best_idx]["title"]
        best_score = scores[best_idx]
        print(f"[{start}:{start+chunk_size}] {best_title=}, {best_score=}")
        if best_title in titles and (i - title_to_last_chunk_idx.get(best_title, -5)) <= 2:
            titles[best_title] = max(titles[best_title], best_score) * 1.2
        else:
            titles[best_title] = best_score
        title_to_last_chunk_idx[best_title] = i

    print(f"{titles=}")
    duration = get_duration(filepath)
    if duration > 3 * 60:
        final_titles = [title for title, score in titles.items() if score > 0.7]
    else:
        best_title = max(titles, key=titles.get)
        final_titles = [best_title] if titles[best_title] > 0.7 else []
    return final_titles

In [ ]:
root = "/mnt/NextcloudSacmData/sacm.av/files/Recordings/Past Events/70th ann. rec"
for f in os.listdir(root):
    final_titles = get_titles(f"{root}/{f}")
    print(f"{d}/{f}: {'_'.join(final_titles)}")

In [ ]:
for d in ["2026-06-27", "2026-06-20", "2026-06-14", "2026-05-07", "2026-04-11", "2026-03-08", "2026-03-07", "2026-02-14", "2026-02-08", "2026-02-07", "2026-02-01", "2026-01-25"]:
    matches = match_zoom_to_sq(d, verbose=False)
    print(d, json.dumps(matches, indent=2, ensure_ascii=False))